In [ ]:
import chromadb

client = chromadb.Client()
collection_name = 'my_collection'
collection = client.get_or_create_collection(collection_name)
docs = [
    {'id':'doc1','text':'Hello World'},
    {'id':'doc2','text':'How are you today?'},
    {'id':'doc3','text':'Good'},
]
for doc in docs:
    collection.upsert(ids=doc['id'],
                      documents=doc['text'])

query = 'Hello World'


/home/thetpaingsoe/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [03:13<00:00, 431kiB/s] 


In [4]:
result = collection.query(
    query_texts=[query],
    n_results=1
)

In [5]:
result

{'ids': [['doc1']],
 'embeddings': None,
 'documents': [['Hello World']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None]],
 'distances': [[0.0]]}

In [1]:
from langchain_core.documents import Document

SAMPLE_DOCS = [
    Document(
        page_content="LangChain is a framework for developing applications powered by language models.",
        metadata={"source": "langchain_docs", "topic": "overview"},
    ),
    Document(
        page_content="LangGraph is a library for building stateful, multi-actor applications with LLMs.",
        metadata={"source": "langgraph_docs", "topic": "overview"},
    ),
    Document(
        page_content="Vector stores are databases optimized for storing and searching embeddings.",
        metadata={"source": "vector_guide", "topic": "database"},
    ),
    Document(
        page_content="RAG combines retrieval with generation for more accurate LLM responses.",
        metadata={"source": "rag_guide", "topic": "architecture"},
    ),
    Document(
        page_content="Embeddings convert text into numerical vectors for semantic similarity.",
        metadata={"source": "embeddings_guide", "topic": "fundamentals"},
    ),
    Document(
        page_content="Chroma is an open-source embedding database for AI applications.",
        metadata={"source": "chroma_docs", "topic": "database"},
    ),
    Document(
        page_content="FAISS is a library for efficient similarity search developed by Facebook.",
        metadata={"source": "faiss_docs", "topic": "database"},
    ),
    Document(
        page_content="Pinecone is a managed vector database service for production workloads.",
        metadata={"source": "pinecone_docs", "topic": "database"},
    ),
]

In [3]:
import tempfile

from dotenv import load_dotenv
from langchain_chroma import Chroma
from sentence_transformers import SentenceTransformer

load_dotenv()

/home/thetpaingsoe/playground/llmBasic/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')   

def similarity_search_with_score():
    with tempfile.TemporaryDirectory() as TempDir:
        vector_store = Chroma.from_documents(
            documents=SAMPLE_DOCS,embedding=embedding,persist_directory=TempDir
        )

        query = 'what is vectors scores?'
        result_with_scores  = vector_store.similarity_search_with_score(query=query,k=3)

        for i,(doc,score) in enumerate(result_with_scores):
            final_score = 1 / (1 + score)
            print(f'Doc {i+1} - Content {doc.page_content} --- Similarity {final_score:.4f} ---  {doc.metadata['source']}')



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17377.85it/s]


In [13]:

similarity_search_with_score()

Doc 1 - Content Vector stores are databases optimized for storing and searching embeddings. --- Similarity 0.4470 ---  vector_guide
Doc 2 - Content Embeddings convert text into numerical vectors for semantic similarity. --- Similarity 0.4241 ---  embeddings_guide
Doc 3 - Content Pinecone is a managed vector database service for production workloads. --- Similarity 0.4013 ---  pinecone_docs


In [16]:
filter_criteria = {'topic':'database'}
def similarity_search_with_filter():
    with tempfile.TemporaryDirectory() as TempDir:
        vector_store = Chroma.from_documents(
            documents=SAMPLE_DOCS,embedding=embedding,persist_directory=TempDir
        )

        query = "What databases are available?"

        results = vector_store.similarity_search(query, k=5)
        print(f"Results without metadata filtering for query '{query}':")
        for i, doc in enumerate(results):
            print(
                f"Doc {i+1} - Content - {doc.page_content} (Source: {doc.metadata['source']} --- {doc.metadata['topic']})"
            )

        results = vector_store.similarity_search(query=query,k=3,filter=filter_criteria)

        for i,doc in enumerate(results):
            print(f'Doc {i+1} - Content - {doc.page_content} ---  {doc.metadata['source']}--- {doc.metadata['topic']}')


In [17]:
similarity_search_with_filter()

Results without metadata filtering for query 'What databases are available?':
Doc 1 - Content - Vector stores are databases optimized for storing and searching embeddings. (Source: vector_guide --- database)
Doc 2 - Content - Chroma is an open-source embedding database for AI applications. (Source: chroma_docs --- database)
Doc 3 - Content - Pinecone is a managed vector database service for production workloads. (Source: pinecone_docs --- database)
Doc 4 - Content - LangGraph is a library for building stateful, multi-actor applications with LLMs. (Source: langgraph_docs --- overview)
Doc 5 - Content - FAISS is a library for efficient similarity search developed by Facebook. (Source: faiss_docs --- database)
Doc 1 - Content - Vector stores are databases optimized for storing and searching embeddings. ---  vector_guide--- database
Doc 2 - Content - Chroma is an open-source embedding database for AI applications. ---  chroma_docs--- database
Doc 3 - Content - Pinecone is a managed vector 

In [22]:
def persist_db():
    db = 'chromadb/'

    vectorstore = Chroma.from_documents(
        documents=SAMPLE_DOCS,embedding=embedding,persist_directory=db
    )


    original_count = vectorstore._collection.count()
    print(f"Persisted vector store with {original_count} documents.")
    print(f"Vector store persisted at: {db}")

    # simulate restart - load from disk
    del vectorstore

    reloaded = Chroma(
        embedding_function=embedding,
        persist_directory=db,
    )

    reloaded_count = reloaded._collection.count()
    print(f"Reloaded vector store with {reloaded_count} documents.")

    retriever = reloaded.as_retriever(
        search_type = 'similarity', search_kwargs = {'k':3}
    )
    result = retriever.invoke('How do i build AI app.')
    for i,doc in enumerate(result):
        print(f'Doc {i+1} -- Content --- {doc.page_content} -- Source {doc.metadata}')


In [23]:
persist_db()

Persisted vector store with 24 documents.
Vector store persisted at: chromadb/
Reloaded vector store with 24 documents.
Doc 1 -- Content --- Chroma is an open-source embedding database for AI applications. -- Source {'source': 'chroma_docs', 'topic': 'database'}
Doc 2 -- Content --- Chroma is an open-source embedding database for AI applications. -- Source {'source': 'chroma_docs', 'topic': 'database'}
Doc 3 -- Content --- Chroma is an open-source embedding database for AI applications. -- Source {'source': 'chroma_docs', 'topic': 'database'}


In [37]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def create_retriever(text:list[str],chunk_size=500,chunk_overlap=50,k=3):
    doc = [Document(page_content=t) for t in text]
    db = 'chromadb'
    spilliter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    split_doc = spilliter.split_documents(doc)
    vector_store = Chroma.from_documents(
        documents=split_doc,embedding=embedding,persist_directory=db
    )

    retriever = vector_store.as_retriever(search_type='similarity',search_kwargs={'k':k})
    return retriever

sample_texts = [
        "Python is a versatile programming language used in web development, "
        "data science, machine learning, and automation. It has a simple syntax "
        "that makes it easy to learn and read.",
        "JavaScript is the language of the web. It runs in browsers and on "
        "servers with Node.js. Modern frameworks like React and Vue make "
        "building web applications efficient.",
        "Rust is a systems programming language focused on safety and "
        "performance. It prevents common bugs like null pointer dereferences "
        "and data races at compile time.",
    ]

retriever = create_retriever(text=sample_texts,chunk_size=200, chunk_overlap=20, k=1)

print('Testing retriever !!!!!!!!!')
queries = [
        "What's good for web development?",
        "Which language is safest?",
        'easy to learn and read'
    ]
for q in queries:
    res = retriever.invoke(q)
    for doc in res:
        print(f" for query {q} - {doc.page_content[:60]}...")
    print()
    print()

Testing retriever !!!!!!!!!
 for query What's good for web development? - JavaScript is the language of the web. It runs in browsers a...


 for query Which language is safest? - JavaScript is the language of the web. It runs in browsers a...


 for query easy to learn and read - Python is a versatile programming language used in web devel...




In [34]:
retriever.invoke("Which language is systems programming ?")

[Document(id='170a8a80-590e-483e-9e69-026291c7be88', metadata={}, page_content='Rust is a systems programming language focused on safety and performance. It prevents common bugs like null pointer dereferences and data races at compile time.')]